<a href="https://colab.research.google.com/github/ksuaray/M4DS/blob/MATH-170-Spring-2026/Lab6_Gradient_Descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MATH 170 — Lab: AD6 Gradient Descent  
## Step-by-Step Through the Lecture Notes

Welcome back to Jupyter — today we’ll use Python, SymPy, NumPy, and plots to walk through the main ideas from **AD6: Gradient Descent** one step at a time.

## Big Idea
In calculus, we learn that minima often occur at **critical numbers**, where

$$
f'(w)=0.
$$

But in data science, we often cannot solve that equation exactly or efficiently. Instead, we use an **iterative method**.


## What you will practice
By the end of this lab, you should be able to:

- compute derivatives with **SymPy**
- perform several iterations of gradient descent by hand and in code
- explain how the **sign of the derivative** determines direction of movement
- interpret a **learning rate**
- recognize common **stopping rules**
- connect gradient descent to **critical numbers**, **minima**, and **optimization**

## Workflow for today
We’ll follow the same structure as the AD6 lecture notes:

1. Start with a function  
2. Compute its derivative  
3. Choose an initial guess and learning rate  
4. Perform iterations  
5. Track the sequence numerically and visually  
6. Interpret what the algorithm is doing

## VARIOSA in This Lab

**V — Visualization**  
- Plot the function and watch how points move toward the minimum  
- Visualize the loss curve and iterative steps  

**A — Approximation**  
- Use step-by-step updates to approximate the minimum  
- Recognize that we are not solving exactly, but getting closer each iteration  

**R — Randomization**  
- (Optional extension) Try different starting points $x_0$ and observe different paths  
- See how initialization affects convergence  

**I — Iteration**  
- Apply the update rule  
$$
w_{n+1} = w_n - \eta f'(w_n)
$$  
repeatedly  
- Track how each step improves the estimate  

**O — Optimization**  
- Minimize a function $f(w)$ using gradient descent  
- Interpret this as finding the “best” value  

**S — Symbolic Manipulation**  
- Use SymPy to compute $f'(w)$ exactly  
- Connect symbolic derivatives to numerical updates  

**A — Aggregation**  
- Summarize iterations in tables (e.g., $w_n$, $f(w_n)$, $f'(w_n)$)  
- Observe patterns and convergence behavior  

## GTW Reminder
When you see **GTW**, that means **Go To Work**:
- write code
- fill in a response
- make a prediction before running the next cell

Let’s begin by importing what we need. Note we're importing a new library, `IPyWidgets` to create an interactive encironment for experimentation.

In [ ]:
# Part 0 — Setup (run this first)
import numpy as np
import sympy as sp
import pandas as pd
import plotly.express as px

#New toys!
import ipywidgets as widgets
from IPython.display import display, clear_output

sp.init_printing()

# Symbols
#x = sp.Symbol('x', real=True)
w = sp.Symbol('w', real=True)

print("Ready!")


##Part 1: Define and Sketch the Function

Use the following cell to define the function you want to optimize. **Make sure you #comment out the other functions!**

In [ ]:
#func_expr = (w**3)*sp.sin(w) # Define the symbolic function
#func_expr = sp.exp(-w +-sp.exp(-w))
#func_expr = 2*x**4-15*x**3+14*x

In the code below, you will need to adjust the window `w_vals` depending on the function to make sure you can view its extrema clearly.

In [ ]:
w_vals = np.linspace(1, 7, 500) # Define a range for w

# Lambdify the SymPy expression to create a numerical function
func_num = sp.lambdify(w, func_expr, 'numpy')

y_vals = func_num(w_vals) # Calculate y values

df = pd.DataFrame({"w": w_vals, "y": y_vals})
fig = px.line(df, x="w", y="y", title="Plot of $f(w)$")
fig.show()

## Part 2: Experiment with Gradient Descent Using Code

Let's create an interactive script to implement gradient descent one iteration at a time. Let's verify the first couple iterations by hand.

In [ ]:
import sympy as sp
import numpy as np


# Define the function and its derivative

deriv_expr = sp.diff(func_expr, w)

# Lambdify for numerical evaluation
f_numeric = sp.lambdify(w, func_expr, 'numpy')
df_numeric = sp.lambdify(w, deriv_expr, 'numpy')

# --- Widgets ---
initial_guess_input = widgets.FloatText(value=1.0, description='Initial Guess:')
learning_rate_input = widgets.FloatText(value=0.01, description='Learning Rate:')
start_reset_button = widgets.Button(description='Start/Reset Gradient Descent')
next_iteration_button = widgets.Button(description='Next Iteration', disabled=True)
output_widget = widgets.Output()

# --- State variables ---
current_w_val = None
current_iteration = 0

# --- Functions ---
def reset_gradient_descent(_):
    global current_w_val, current_iteration
    current_w_val = initial_guess_input.value
    current_iteration = 0
    next_iteration_button.disabled = False
    with output_widget:
        clear_output()
        print(f"Gradient Descent for f(w) = {func_expr}")
        print(f"Derivative df/dw = {deriv_expr}")
        print(f"Initial Guess: {initial_guess_input.value}, Learning Rate: {learning_rate_input.value}")
        print("-" * 40)
    # Automatically run the first iteration after reset
    run_next_iteration(None)


def run_next_iteration(_):
    global current_w_val, current_iteration

    if current_w_val is None:
        with output_widget:
            print("Please click 'Start/Reset Gradient Descent' first.")
        return

    current_iteration += 1
    f_val = f_numeric(current_w_val)
    df_val = df_numeric(current_w_val)

    with output_widget:
        print(f"\nIteration {current_iteration}:")
        print(f"  Current w: {current_w_val:.6f}")
        print(f"  f(w): {f_val:.6f}")
        print(f"  df/dx: {df_val:.6f}")

        # Check for convergence (derivative close to zero)
        if abs(df_val) < 1e-4:
            print("\nConverged to a local extremum!")
            next_iteration_button.disabled = True
            return

        next_w = current_w_val - learning_rate_input.value * df_val
        print(f"  Next w: {next_w:.6f}")

        # Check for very small step size (also indicates convergence)
        if current_iteration > 1 and abs(next_w - current_w_val) < 1e-6:
             print("\nStep size is very small, likely converged.")
             next_iteration_button.disabled = True
             return

        current_w_val = next_w

        # Safety break for too many iterations
        if current_iteration > 500:
            print("\nMaximum iterations reached. Consider adjusting learning rate or initial guess.")
            next_iteration_button.disabled = True
            return

# --- Event Handlers ---
start_reset_button.on_click(reset_gradient_descent)
next_iteration_button.on_click(run_next_iteration)

# --- Display UI ---
ui = widgets.VBox([
    widgets.HBox([initial_guess_input, learning_rate_input]),
    widgets.HBox([start_reset_button, next_iteration_button]),
    output_widget
])

display(ui)

Let's compare our approximation to the actual extreme value:

In [ ]:
import sympy as sp
import numpy as np

# Ensure w is defined and func_expr is available from previous cells
# x = sp.Symbol('x', real=True)
# func_expr = 2*x**4 - 15*x**3 + 14*x
# w_vals = np.linspace(-2, 2, 500) # Defined in previous cell

# Get the derivative of func_expr
deriv_expr = sp.diff(func_expr, w)

# Lambdify for numerical evaluation of the function and its derivative
f_numeric = sp.lambdify(w, func_expr, 'numpy')
df_numeric = sp.lambdify(w, deriv_expr, 'numpy') # Numerical derivative

# Define the bounds of w_vals
w_min_bound = w_vals.min()
w_max_bound = w_vals.max()

# Collect candidate w values: critical points within bounds and boundary points
candidate_w_values = set()

# Add boundary points
candidate_w_values.add(w_min_bound)
candidate_w_values.add(w_max_bound)

# --- Numerical search for critical points ---
# Evaluate the derivative over the w_vals range
df_vals = df_numeric(w_vals)

# Find where the derivative changes sign (indicating a root)
# This finds indices where df_vals[i] and df_vals[i+1] have opposite signs
# or where df_vals[i] is very close to zero.
for i in range(len(w_vals) - 1):
    # Check for sign change
    if df_vals[i] * df_vals[i+1] < 0:
        # Approximate the root by taking the midpoint of the interval
        approx_root = (w_vals[i] + w_vals[i+1]) / 2
        candidate_w_values.add(approx_root)
    # Also check if the derivative is very close to zero directly
    elif abs(df_vals[i]) < 1e-6: # Using a small tolerance
        candidate_w_values.add(w_vals[i])

# Convert set to sorted list
candidate_w_values = sorted(list(candidate_w_values))

# Evaluate function at candidate w values
f_values_at_candidates = [f_numeric(val) for val in candidate_w_values]

# Find the overall minimum and maximum
# This finds global min/max within the candidate points and boundary
if f_values_at_candidates: # Ensure list is not empty
    min_value = min(f_values_at_candidates)
    max_value = max(f_values_at_candidates)

    min_w_index = f_values_at_candidates.index(min_value)
    max_w_index = f_values_at_candidates.index(max_value)

    min_w = candidate_w_values[min_w_index]
    max_w = candidate_w_values[max_w_index]

    print(f"Function expression: {func_expr}")
    print(f"Derivative expression: {deriv_expr}")
    # Update print statement for critical points to reflect numerical finding
    # Filter out boundary points from this display for clarity, but they are in candidate_w_values
    numerical_critical_points_display = [cp for cp in candidate_w_values if cp != w_min_bound and cp != w_max_bound]
    print(f"Numerically found critical points (approx): {numerical_critical_points_display}")
    print(f"Considered w-range: [{w_min_bound}, {w_max_bound}]")
    print(f"Candidate w values for extrema (including boundaries): {candidate_w_values}")
    print(f"Function values at candidates: {f_values_at_candidates}")

    print(f"\nGlobal Minimum in range: f({min_w:.4f}) = {min_value:.4f}")
    print(f"Global Maximum in range: f({max_w:.4f}) = {max_value:.4f}")
else:
    print("No candidate w values found (either no critical points in range or an empty range after filtering).")


# **LAB 6 Homework**

Use the code above to complete Example III in the AD6 Lecture Notes. You can use a LLM to produce your table here by copy-pasting your table and asking the AI to provide the **markdown** version of your table.

Provide the table below: